In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep
hep.style.use("CMS")
from coffea import util
import itertools
import os

## Scale factors and IOV

In [ ]:
IOVs = ['2016APV', '2016']

lumi = {
    "2016APV": 19800.,
    "2016": 16120., #35920 - 19800
    "2016all": 35920,
    "2017": 41530.,
    "2018": 59740.
}

t_BR = 0.6741
ttbar_BR = 0.4544 #PDG 2019
ttbar_xs1 = 831.76 * (0.09210) #pb For ttbar mass from 700 to 1000
ttbar_xs2 = 831.76 * (0.02474) #pb For ttbar mass from 1000 to Inf
toptag_sf = 0.9
toptag_kf = 0.7 #0.7
qcd_xs = 1370000000.0 #pb From https://cms-gen-dev.cern.ch/xsdb



## make plot image filenames

In [ ]:
directories = [
    'images/png/discriminatorPlots/2016all',
    'images/png/discriminatorPlots/2016APV',
    'images/png/discriminatorPlots/2016',
    'images/png/discriminatorPlots/2017',
    'images/png/discriminatorPlots/2018',
    'images/pdf/discriminatorPlots/2016all',
    'images/pdf/discriminatorPlots/2016APV',
    'images/pdf/discriminatorPlots/2016',
    'images/pdf/discriminatorPlots/2017',
    'images/pdf/discriminatorPlots/2018',
]


for path in directories:
    if not os.path.exists(path):
        os.makedirs(path)

## coffea files

In [ ]:
coffea_dir = '../outputs/'
coffeaFiles = {
    "JetHT":{
        "2016APV": {
            "B": coffea_dir+'JetHT_2016APVB.coffea',
            "C": coffea_dir+'JetHT_2016APVC.coffea',
            "D": coffea_dir+'JetHT_2016APVD.coffea',
            "E": coffea_dir+'JetHT_2016APVE.coffea',
            "F": coffea_dir+'JetHT_2016APVF.coffea',
        },
        "2016": {
            "F": coffea_dir+'JetHT_2016F.coffea',
            "G": coffea_dir+'JetHT_2016G.coffea',
            "H": coffea_dir+'JetHT_2016H.coffea',

        },
        "2017": '',
        "2018": ''
    },        
    
    "TTbar": {
        "2016APV": {
            "700to1000": coffea_dir+'TTbar_2016APV_700to1000.coffea',
            "1000toInf": coffea_dir+'TTbar_2016APV_1000toInf.coffea',
        },
        "2016": {
            "700to1000": coffea_dir+'TTbar_2016_700to1000.coffea',
            "1000toInf": coffea_dir+'TTbar_2016_1000toInf.coffea',
        },
        "2017": {
            "700to1000": '',
            "1000toInf": '',
        },
        "2018": {
            "700to1000": '',
            "1000toInf": '',
        }
    }  
}

## load histograms as function of mass

In [ ]:
hist = 'deepAK8_mtt'
histograms = {}

for IOV in IOVs:
    
    
    jethtfiles = []
    for key, file in coffeaFiles["JetHT"][IOV].items():
        jethtfiles.append(util.load(file))
    
    ttbarfiles = [util.load(coffeaFiles["TTbar"][IOV]['700to1000']), util.load(coffeaFiles["TTbar"][IOV]['1000toInf'])]
    
    hist_datas = {
        "[0.0, 0.2)":[],
        "[0.2, 0.4)":[],
        "[0.4, 0.6)":[],
        "[0.6, 0.8)":[],
        "[0.8, 1.0)":[],
        "pt":[],
        "mass":[],
        "rho":[]
    }
    hist_ttbars = {
        "[0.0, 0.2)":[],
        "[0.2, 0.4)":[],
        "[0.4, 0.6)":[],
        "[0.6, 0.8)":[],
        "[0.8, 1.0)":[],
        "pt":[],
        "mass":[],
        "rho":[]
    }
    
    # 'deepAK8' : hist.Hist(jetpt_axis, ttbarmass_axis, m_pT_axis, deepAK8_axis, storage="weight", name="Counts") 
    for coffeafile in jethtfiles:
        hist_datas["[0.0, 0.2)"].append(coffeafile[hist][sum, :, sum, 0])
        hist_datas["[0.2, 0.4)"].append(coffeafile[hist][sum, :, sum, 10])
        hist_datas["[0.4, 0.6)"].append(coffeafile[hist][sum, :, sum, 20])
        hist_datas["[0.6, 0.8)"].append(coffeafile[hist][sum, :, sum, 30])
        hist_datas["[0.8, 1.0)"].append(coffeafile[hist][sum, :, sum, 40])
        for i in range(1,10):
            hist_datas["[0.0, 0.2)"].append(coffeafile[hist][sum, :, sum, i])
            hist_datas["[0.2, 0.4)"].append(coffeafile[hist][sum, :, sum, 10+i])
            hist_datas["[0.4, 0.6)"].append(coffeafile[hist][sum, :, sum, 20+i])
            hist_datas["[0.6, 0.8)"].append(coffeafile[hist][sum, :, sum, 30+i])
            hist_datas["[0.8, 1.0)"].append(coffeafile[hist][sum, :, sum, 40+i])   
        hist_datas["pt"].append(coffeafile[hist][:, sum, sum, :])
        hist_datas["mass"].append(coffeafile[hist][sum, :, sum, :])
        hist_datas["rho"].append(coffeafile[hist][sum, sum, :, :])
        
    for coffeafile in ttbarfiles:
        hist_ttbars["[0.0, 0.2)"].append(coffeafile[hist][sum, :, sum, 0])
        hist_ttbars["[0.2, 0.4)"].append(coffeafile[hist][sum, :, sum, 10])
        hist_ttbars["[0.4, 0.6)"].append(coffeafile[hist][sum, :, sum, 20])
        hist_ttbars["[0.6, 0.8)"].append(coffeafile[hist][sum, :, sum, 30])
        hist_ttbars["[0.8, 1.0)"].append(coffeafile[hist][sum, :, sum, 40])
        for i in range(1,10):
            hist_ttbars["[0.0, 0.2)"].append(coffeafile[hist][sum, :, sum, i])
            hist_ttbars["[0.2, 0.4)"].append(coffeafile[hist][sum, :, sum, 10+i])
            hist_ttbars["[0.4, 0.6)"].append(coffeafile[hist][sum, :, sum, 20+i])
            hist_ttbars["[0.6, 0.8)"].append(coffeafile[hist][sum, :, sum, 30+i])
            hist_ttbars["[0.8, 1.0)"].append(coffeafile[hist][sum, :, sum, 40+i])
        hist_ttbars["pt"].append(coffeafile[hist][:, sum, sum, :])
        hist_ttbars["mass"].append(coffeafile[hist][sum, :, sum, :])
        hist_ttbars["rho"].append(coffeafile[hist][sum, sum, :, :])
    
    
    # add together data hist eras
    hist_data = {}
    for b,hists in hist_datas.items():
        hist_data[b] = hists[0]
        for i in range(len(hists) - 1): 
            hist_data[b] += hists[i+1]

    # add together ttbar hist regions
    hist_ttbar = {}
    for b,hists in hist_ttbars.items():
        hist_ttbar[b] = hists[0] + hists[1]  
    
    histograms[IOV] = {
                        "data":  hist_data,
                        "ttbar": hist_ttbar
                    }
# print(histograms['2016APV']['data']['0-2'])

## plot histograms with mass as x-axis

In [ ]:
IOV = '2016all'
tagger = 'DeepAK8'
allbins = False

fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(20,7))
hdata = {}
httbar = {}
if IOV == '2016all':
    for b,hists in hist_datas.items():
        hdata[b]  = histograms['2016APV']['data'][b]  + histograms['2016']['data'][b]
        httbar[b] = histograms['2016APV']['ttbar'][b] + histograms['2016']['ttbar'][b]
    year = '2016'
else:
    for b,hists in hist_datas.items():
        hdata[b]  = histograms[IOV]['data'][b]  
        httbar[b] = histograms[IOV]['ttbar'][b] 
    year = IOV
     

hep.cms.label('', data=True, lumi='{0:0.1f}'.format(lumi[IOV]/1000.), year=year, loc=3, fontsize=20, ax=ax1)
hep.cms.label('', data=False, lumi='{0:0.1f}'.format(lumi[IOV]/1000.), year=year, loc=3, fontsize=20, ax=ax2)
hep.cms.text('Preliminary', loc=3, fontsize=20, ax=ax1)

for b,hists in hist_datas.items():
    if "[0." in b: # Only plot 1D hists
        continue
#         # ---------------- Integrations ---------------- #
#         dataNorm = np.sum(hdata[b].values())
#         ttbarNorm = np.sum(httbar[b].values())
#         # ---------------- Scaling to Unity ---------------- #
#         hdata[b] *= 1./dataNorm
#         httbar[b] *= 1./ttbarNorm
#         # ---------------- Plot ---------------- #
#         hep.histplot(hdata[b],  ax=ax1, histtype='step', label=f'Data; {tagger} {b}')
#         hep.histplot(httbar[b], ax=ax2, histtype='step', label=f'TTbar; {tagger} {b}')
    elif 'mass' in b:
        for i in range(2):
            for j in range(5):
                denom = hdata[b][i, j].value
                print(j)
                for k in range (1,10):
                    denom += hdata[b][i, j*10+k].value
                print(1./denom)
                hdata[b] *= (1./denom)
                hep.hist2dplot(hdata[b], ax=ax1, vmax=0.8)
                ax1.set_xlim(800+144*i, 800+144*(i+1))
                ax1.set_ylim(0.0+0.2*j, 0.0+0.2*(j+1))
                plt.show()

# i = 800
# while (i < 1000):
# map2d = hep.hist2dplot(h2ddata["mass"])
# print(h2ddata["mass"].density()[4])
#     dataNorm2d = 1./np.sum(hep.hist2dplot(h2ddata["mass"])[0].get_array().data)
#     map2d *= dataNorm2d
#     ax1.set_xlim(i, i+144)
#     ax1.set_ylim(0.2, 0.4)
#     print(i)
#     print(h2ddata["mass"])
    
#     i += 144
#     plt.show()

# ax1.legend()
# ax2.legend()

# if 'mtt' in hist:
#     ax1.set_ylim(0, 0.175)
#     ax1.set_xlim(900, 4000)
#     ax2.set_ylim(0, 0.200)
#     ax2.set_xlim(900, 4000)
#     plt.savefig(f'images/png/discriminatorPlots/{IOV}/mtt.png')
#     plt.savefig(f'images/pdf/discriminatorPlots/{IOV}/mtt.pdf')
# elif 'msd' in hist:
#     if allbins:
#         ax1.set_xlim(0, 300)
#         ax2.set_xlim(0, 300)
#         plt.savefig(f'images/png/discriminatorPlots/{IOV}/msd.png')
#         plt.savefig(f'images/pdf/discriminatorPlots/{IOV}/msd.pdf')
#     else:
#         ax1.set_ylim(0, 0.08)
#         ax1.set_xlim(20, 300)
#         ax2.set_ylim(0, 0.16)
#         ax2.set_xlim(20, 300)
#         plt.savefig(f'images/png/discriminatorPlots/{IOV}/msd_twoBinsRemoved.png')
#         plt.savefig(f'images/pdf/discriminatorPlots/{IOV}/msd_twoBinsRemoved.pdf')
# plt.show()

## load histograms as function of $\rho\ =\ m/p_T$

In [ ]:
hist = 'deepAK8_msd'
histograms = {}

for IOV in IOVs:
    
    
    jethtfiles = []
    for key, file in coffeaFiles["JetHT"][IOV].items():
        jethtfiles.append(util.load(file))
    
    ttbarfiles =  {
        "700to1000": util.load(coffeaFiles["TTbar"][IOV]['700to1000']),
        "1000toInf": util.load(coffeaFiles["TTbar"][IOV]['1000toInf'])
    }
    
    
    hist_datas = {
        "[0.0, 0.2)":[],
        "[0.2, 0.4)":[],
        "[0.4, 0.6)":[],
        "[0.6, 0.8)":[],
        "[0.8, 1.0)":[]
    }
    
    # 'deepAK8' : hist.Hist(jetpt_axis, ttbarmass_axis, m_pT_axis, deepAK8_axis, storage="weight", name="Counts") 
    for coffeafile in jethtfiles:
        hist_datas["[0.0, 0.2)"].append(coffeafile[hist][sum, sum, :, 0])
        hist_datas["[0.2, 0.4)"].append(coffeafile[hist][sum, sum, :, 10])
        hist_datas["[0.4, 0.6)"].append(coffeafile[hist][sum, sum, :, 20])
        hist_datas["[0.6, 0.8)"].append(coffeafile[hist][sum, sum, :, 30])
        hist_datas["[0.8, 1.0)"].append(coffeafile[hist][sum, sum, :, 40])
        
        
    hist_ttbars = {
        "[0.0, 0.2)":[
            ttbarfiles['700to1000'][hist][sum, sum, :, 0],
            ttbarfiles['1000toInf'][hist][sum, sum, :, 0]
        ],
        "[0.2, 0.4)":[
            ttbarfiles['700to1000'][hist][sum, sum, :, 10],
            ttbarfiles['1000toInf'][hist][sum, sum, :, 10]
        ],
        "[0.4, 0.6)":[
            ttbarfiles['700to1000'][hist][sum, sum, :, 20],
            ttbarfiles['1000toInf'][hist][sum, sum, :, 20]
        ],
        "[0.6, 0.8)":[
            ttbarfiles['700to1000'][hist][sum, sum, :, 30],
            ttbarfiles['1000toInf'][hist][sum, sum, :, 30]
        ],
        "[0.8, 1.0)":[
            ttbarfiles['700to1000'][hist][sum, sum, :, 40],
            ttbarfiles['1000toInf'][hist][sum, sum, :, 40]
        ]
    }
    
    
    # add together data hist eras
    hist_data = {}
    for b,hists in hist_datas.items():
        hist_data[b] = hists[0]
        for i in range(len(hists) - 1): 
            hist_data[b] += hists[i+1]

    # add together ttbar hist regions
    hist_ttbar = {}
    for b,hists in hist_ttbars.items():
        hist_ttbar[b] = hists[0] + hists[1]    
    
    histograms[IOV] = {
                        "data":  hist_data,
                        "ttbar": hist_ttbar
                    }
# print(histograms['2016APV']['data']['0-2'])

## plot histograms with mass/pT as x-axis

In [ ]:
IOV = '2016all'
tagger = 'DeepAK8'
allbins = False

fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(20,7))
hdata = {}
httbar = {}
if IOV == '2016all':
    for b,hists in hist_datas.items():
        hdata[b]  = histograms['2016APV']['data'][b]  + histograms['2016']['data'][b]
        httbar[b] = histograms['2016APV']['ttbar'][b] + histograms['2016']['ttbar'][b]
    year = '2016'
else:
    for b,hists in hist_datas.items():
        hdata[b]  = histograms[IOV]['data'][b]  
        httbar[b] = histograms[IOV]['ttbar'][b] 
    year = IOV
     

hep.cms.label('', data=True, lumi='{0:0.1f}'.format(lumi[IOV]/1000.), year=year, loc=3, fontsize=20, ax=ax1)
hep.cms.label('', data=True, lumi='{0:0.1f}'.format(lumi[IOV]/1000.), year=year, loc=3, fontsize=20, ax=ax2)
hep.cms.text('Preliminary', loc=3, fontsize=20, ax=ax1)
hep.cms.text('Preliminary', loc=3, fontsize=20, ax=ax2)

for b,hists in hist_datas.items():
    # ---------------- Integrations ---------------- #
    dataNorm = np.sum(hdata[b].values())
    ttbarNorm = np.sum(httbar[b].values())
    # ---------------- Scaling to Unity ---------------- #
    hdata[b] *= 1./dataNorm
    httbar[b] *= 1./ttbarNorm
    # ---------------- Plot ---------------- #
    hep.histplot(hdata[b],  ax=ax1, histtype='step', label=f'Data; {tagger} {b}')
    hep.histplot(httbar[b], ax=ax2, histtype='step', label=f'TTbar; {tagger} {b}')
    

ax1.legend()
ax2.legend()

if 'mtt' in hist:
    ax1.set_ylim(0, 0.2)
    ax2.set_ylim(0, 0.2)
    plt.savefig(f'images/png/discriminatorPlots/{IOV}/rho.png')
    plt.savefig(f'images/pdf/discriminatorPlots/{IOV}/rho.pdf')
elif 'msd' in hist:
    if allbins:
        plt.savefig(f'images/png/discriminatorPlots/{IOV}/rho_sd.png')
        plt.savefig(f'images/pdf/discriminatorPlots/{IOV}/rho_sd.pdf')
    else:
        ax1.set_ylim(0, 0.08)
        ax1.set_xlim(0.04, 0.5)
        ax2.set_ylim(0, 0.2)
        ax2.set_xlim(0.04, 0.5)
        plt.savefig(f'images/png/discriminatorPlots/{IOV}/rho_sd_twoBinsRemoved.png')
        plt.savefig(f'images/pdf/discriminatorPlots/{IOV}/rho_sd_twoBinsRemoved.pdf')
        
plt.show()